In [1]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import numpy as np
# from sklearn.metrics import precision_recall_fscore_support
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Arial"
import seaborn as sns
# from joblib import Parallel, delayed
import re
import os
from itertools import combinations
# from statsmodels.stats.multitest import multipletests

plt.rcParams["font.family"] = "Arial"


The purpose of this notebook is to create

- Figure 2a: a bar plot comparing the performance (precision and recall) of the models on the cogntive status tasks (NC/MCI/DE), with stats using bootstrap.
- A table containing the same data, plus the F1 score. The table is in LaTeX format, ready to be pasted into the manuscript.

Notice that we are only loading the COG task

In [2]:
multilabel_cases = pd.read_csv("/projectnb/vkolagrp/projects/adrd_foundation_model/benchmarks/nacc_multilabel/nacc_multilabel_test_corrected.csv")

In [3]:
cols = ['AD', 'FTLD', 'LBD', 'ODE', 'PSY', 'SEF', 'VD']
multilabel_cases['non_zero_cols'] = multilabel_cases[cols].astype(bool).sum(axis=1)

In [4]:
multilabel_cases.head()

,NACCID,visit_summary,diagnosis,AD,FTLD,LBD,ODE,PSY,SEF,VD,options,question,ETPR,non_zero_cols
0,NACC282885,The subject is an 83-year-old right-handed mal...,NC,0,0,0,0,0,0,0,A. Psychiatric conditions including schizophre...,Identify all the etiologes of the patient's co...,NC,0
1,NACC837281,"The subject is a 64-year-old right-handed, Whi...",DE,0,1,0,0,0,0,0,A. Systemic and environmental factors includin...,Determine all the etiologies underlying the pa...,FTLD,1
2,NACC355961,"The subject is a 77-year-old right-handed, Eng...",MCI,1,0,0,0,0,0,0,A. Systemic and environmental factors includin...,Identify all the etiologes of the patient's co...,AD,1
3,NACC385116,The patient is a 57-year-old White female who ...,NC,0,0,0,0,0,0,0,A. Not applicable (no cognitive impairment)\nB...,Identify all the etiologic diagnoses of the pa...,NC,0
4,NACC260495,The subject is a 70-year-old male of White rac...,DE,0,1,0,0,0,0,0,A. Alzheimer's disease (AD)\nB. Frontotemporal...,"Using the information provided, identify all t...",FTLD,1


In [5]:
ids_with_1_etpr = list(multilabel_cases[multilabel_cases['non_zero_cols'] == 1]['NACCID'])
ids_with_more_etpr = list(multilabel_cases[(multilabel_cases['non_zero_cols'] > 1)]['NACCID'])
len(ids_with_1_etpr), len(ids_with_more_etpr)

(11774, 4195)

In [6]:
nacc_path = Path('/projectnb/vkolagrp/projects/adrd_foundation_model/results/NACC/test_etpr')

The following functions load the data from the paths specified above, pick only relevant columns, and concatenate everything. 

Later we compute class-wise precision and recall, and estimate error bars on them using bootstrap. The bootstrap samples are kept, so we can compute anything else we want after resampling, ensuring that the calculations use the correct samples.

In [7]:
def option_string_to_dict(options):
    # The option string is randomized (e.g. NC is not always option A). We need to break down the 
    # options and look at the text (e.g. MCI), not just the letter that identifies them in a particular question
    pattern = r"([A-Z])\. ([^\n]+)"
    matches = re.findall(pattern, options)
    return {key: value for key, value in matches}

def load_answers(dir_path, dataset_name, id_list, id_type):
    # load all parquet files from the directory, stack them into a pandas datafame
    # this only reads the participant ID, ground trush answer and the prediction. 
    # Reading only those columns is significantly faster (about 100x) than loading the whole dataframe.
    # Loading everything is very slow because there are extremely long strings (model outputs) in some columns

    fpaths = list(dir_path.rglob('*.parquet'))

    dfs = []

    cols_to_read = ['ID','ground_truth','prediction','ground_truth_text','options'] #, 'generated_text']

    for fpath in tqdm(fpaths):

        model = fpath.parent.name.split('-',3)[-1] 
        benchmark = fpath.parent.parent.name.split('_',1)[-1].upper()

        df = pd.read_parquet(fpath,columns=cols_to_read)
    
        df = df.assign(model=model, benchmark=benchmark)
        
        if dataset_name == "NACC":
            df =  df[df['ID'].isin(id_list)].reset_index(drop=True)

        df['correct'] = (df['ground_truth'] == df['prediction']).astype(int)

        df['prediction_text'] = df.apply(lambda row: option_string_to_dict(row['options']).get(row['prediction'],'invalid'),axis=1)
        
        df['id_type'] = id_type

        dfs.append(df)

    df = pd.concat(dfs)
    df['dataset'] = dataset_name

    # make these columns Categorical
    group_cols = ["dataset", "benchmark", "model", "ground_truth_text", 'prediction_text']
    for col in group_cols:
        df[col] = pd.Categorical(df[col])

    # keep only the models we actually care about
    df = df[df['model'].isin(['Qwen2.5-3B-Instruct','Qwen2.5-7B-Instruct','NACC-3B-OS-SCE'])]

    return df


The `dataset_name` parameter will be used in the results dataset to identify which dataset the data came from

In [8]:
nacc_res_1 = load_answers(nacc_path,dataset_name='NACC', id_list=ids_with_1_etpr, id_type="ONE")

  0%|          | 0/8 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:05<00:00,  1.35it/s]


In [9]:
len(nacc_res_1)

176610

In [10]:
nacc_res_many = load_answers(nacc_path,dataset_name='NACC', id_list=ids_with_more_etpr, id_type="MORE_THAN_ONE")

  0%|          | 0/8 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:03<00:00,  2.57it/s]


In [11]:
len(nacc_res_many)

62925

In [12]:
# concatenate everything in a tall format dataframe

results_df = pd.concat(
    [
        nacc_res_1,
        nacc_res_many,
    ]
).reset_index(drop=True)

results_df["trial"] = results_df.index
results_df.head()

,ID,ground_truth,prediction,ground_truth_text,options,model,benchmark,correct,prediction_text,id_type,dataset,trial
0,NACC344354,G,G,Alzheimer's disease (AD),A. Not applicable (no cognitive impairment)\nB...,Qwen2.5-7B-Instruct,ETPR,1,Alzheimer's disease (AD),ONE,NACC,0
1,NACC344354,G,A,Alzheimer's disease (AD),A. Not applicable (no cognitive impairment)\nB...,Qwen2.5-7B-Instruct,ETPR,0,Not applicable (no cognitive impairment),ONE,NACC,1
2,NACC344354,G,D,Alzheimer's disease (AD),A. Not applicable (no cognitive impairment)\nB...,Qwen2.5-7B-Instruct,ETPR,0,Vascular brain injury or vascular dementia inc...,ONE,NACC,2
3,NACC344354,G,A,Alzheimer's disease (AD),A. Not applicable (no cognitive impairment)\nB...,Qwen2.5-7B-Instruct,ETPR,0,Not applicable (no cognitive impairment),ONE,NACC,3
4,NACC344354,G,G,Alzheimer's disease (AD),A. Not applicable (no cognitive impairment)\nB...,Qwen2.5-7B-Instruct,ETPR,1,Alzheimer's disease (AD),ONE,NACC,4


In [13]:
# results_df['dataset_raw'] = results_df['dataset']

# results_df['dataset'] = results_df['dataset'].replace(
#     {
#         'ADNI':'Other',
#         'BrainLat':'Other',
#         'NIFD':'Other',
#         'PPMI':'Other',
#     }
# )

In [14]:
results_df.sample(3)

,ID,ground_truth,prediction,ground_truth_text,options,model,benchmark,correct,prediction_text,id_type,dataset,trial
133174,NACC009277,G,G,Lewy body disease (LBD),A. Psychiatric conditions including schizophre...,Qwen2.5-3B-Instruct,ETPR,1,Lewy body disease (LBD),ONE,NACC,133174
202037,NACC655835,C,E,Alzheimer's disease (AD),A. Frontotemporal lobar degeneration and its v...,NACC-3B-OS-SCE,ETPR,0,Vascular brain injury or vascular dementia inc...,MORE_THAN_ONE,NACC,202037
10537,NACC537844,H,D,Alzheimer's disease (AD),A. Vascular brain injury or vascular dementia ...,Qwen2.5-7B-Instruct,ETPR,0,Systemic and environmental factors including i...,ONE,NACC,10537


In [15]:
results_df['ground_truth_text'].value_counts()

ground_truth_text
Alzheimer's disease (AD)                                                                                                                                                                                             196485
Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTLD)     18870
Lewy body disease (LBD)                                                                                                                                                                                                8925
Vascular brain injury or vascular dementia including stroke (VD)                                                                                                                                                       6045
Systemic and environmental factors including infectious diseases (HIV included), metabolic, substance 

In [16]:
len(results_df)

239535

In [17]:
# ============================================================================
# QUICK CHECK
# ============================================================================

from itertools import combinations

print("=" * 80)
print("CHECKING DATA ALIGNMENT FOR PERMUTATION TESTS")
print("=" * 80)

issues_found = False

for (ds, bench), group in results_df.groupby(["dataset", "benchmark"], observed=True):
    models = sorted(group["model"].unique())
    
    print(f"\n📊 {ds} / {bench}")
    
    for m1, m2 in combinations(models, 2):
        d1 = group[group["model"] == m1].copy()
        d2 = group[group["model"] == m2].copy()
        
        # Sort by ID
        d1_sorted = d1.sort_values('ID').reset_index(drop=True)
        d2_sorted = d2.sort_values('ID').reset_index(drop=True)
        
        # Check if IDs match exactly in the same order
        ids_match = d1_sorted['ID'].equals(d2_sorted['ID'])
        
        # Get differences
        ids1 = set(d1['ID'])
        ids2 = set(d2['ID'])
        only_m1 = ids1 - ids2
        only_m2 = ids2 - ids1
        
        if ids_match:
            print(f"   ✅ {m1} vs {m2}: {len(d1)} samples, perfectly aligned")
        else:
            issues_found = True
            print(f"   ❌ {m1} vs {m2}: MISALIGNED!")
            print(f"      {m1}: {len(d1)} samples")
            print(f"      {m2}: {len(d2)} samples")
            
            if only_m1:
                print(f"      Only in {m1}: {len(only_m1)} IDs")
                if len(only_m1) <= 5:
                    print(f"         {sorted(only_m1)}")
            
            if only_m2:
                print(f"      Only in {m2}: {len(only_m2)} IDs")
                if len(only_m2) <= 5:
                    print(f"         {sorted(only_m2)}")
            
            # Check if same IDs but different order
            if ids1 == ids2 and not ids_match:
                print(f"      ⚠️  SAME IDs but DIFFERENT ORDER - will cause wrong pairing!")

print("\n" + "=" * 80)
if issues_found:
    print("❌ ALIGNMENT ISSUES FOUND - PERMUTATION TESTS MAY BE INVALID!")
    print("   See fix below...")
else:
    print("✅ ALL DATA PROPERLY ALIGNED - PERMUTATION TESTS ARE VALID")
print("=" * 80)

CHECKING DATA ALIGNMENT FOR PERMUTATION TESTS

📊 NACC / ETPR
   ✅ NACC-3B-OS-SCE vs Qwen2.5-3B-Instruct: 79845 samples, perfectly aligned
   ✅ NACC-3B-OS-SCE vs Qwen2.5-7B-Instruct: 79845 samples, perfectly aligned
   ✅ Qwen2.5-3B-Instruct vs Qwen2.5-7B-Instruct: 79845 samples, perfectly aligned

✅ ALL DATA PROPERLY ALIGNED - PERMUTATION TESTS ARE VALID


In [18]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from itertools import combinations
from statsmodels.stats.multitest import multipletests


def _vectorized_metric_calc(y_true, y_pred, label_code, metric):
    """
    Vectorized per-label metric over last axis.
    Works for:
      - bootstrap arrays: y_true/y_pred shape (n_boot, n)
      - single arrays:    y_true/y_pred shape (n,)
    Returns:
      - shape (n_boot,) or scalar/shape (n,) consistent with numpy broadcasting.
    """
    tp = np.sum((y_pred == label_code) & (y_true == label_code), axis=-1)
    if metric == "precision":
        den = np.sum(y_pred == label_code, axis=-1)
    else:  # recall
        den = np.sum(y_true == label_code, axis=-1)
    return np.divide(tp, den, out=np.zeros_like(tp, dtype=float), where=den != 0)

def _vectorized_macro_metric_calc(y_true, y_pred, label_codes, metric):
    """
    Macro average: mean over labels of per-label metric.
    For bootstrap: returns shape (n_boot,)
    For non-bootstrap: returns scalar (float) if input is (n,)
    """
    label_codes = np.asarray(label_codes, dtype=np.int16)
    if label_codes.size == 0:
        # No labels to average; return zeros shaped like per-label output
        return np.zeros(y_true.shape[0], dtype=float) if y_true.ndim > 1 else 0.0

    acc = None
    label_codes_updated = []
    for lbl in label_codes:
        v = _vectorized_metric_calc(y_true, y_pred, int(lbl), metric)
        # print(v, lbl)
        label_codes_updated.append(lbl)
        if acc is None:
            acc = v.astype(float, copy=False)
        else:
            acc = acc + v
    return acc / float(len(label_codes_updated))

In [19]:
def _single_bootstrap_task_macro(group_info, y_true_b, y_pred_b, label_codes, m_type):
    boot_values = _vectorized_macro_metric_calc(y_true_b, y_pred_b, label_codes, m_type)
    low, med, high = np.percentile(boot_values, [2.5, 50, 97.5])
    return {
        **group_info,
        "metric": f"macro_{m_type}",
        "mean": float(np.mean(boot_values)),
        "median": float(med),
        "low": float(low),
        "high": float(high),
    }

def optimized_bootstrap_parallel_macro(df, n_boot=1000, seed=42, n_jobs=-1):
    """
    Bootstrap CIs for macro precision/recall per (dataset, benchmark, model),
    where macro is averaged over the labels PRESENT in that group (excluding 'invalid').
    """
    
    # Keep original text columns for grouping
    df_grouped = df[["ID", "trial", "id_type", "dataset", "benchmark", "model", "ground_truth_text", "prediction_text"]].copy()
    
    # Build tasks (2 per group)
    groups = list(df_grouped.groupby(["id_type", "model"], observed=True))
    main_rng = np.random.default_rng(seed)
    all_tasks = []
    
    print(f"Preparing Bootstrap data for {len(groups) * 2} macro tasks...")
    
    for g_id, group in groups:
        # Sort by ID for consistency
        group = group.sort_values(['ID', 'trial']).reset_index(drop=True)
        
        # Get all unique categories for THIS group
        gt_cats = group["ground_truth_text"].astype("category").cat.categories.tolist()
        pred_cats = group["prediction_text"].astype("category").cat.categories.tolist()
        all_cats = sorted(set(gt_cats + pred_cats))
        
        # Ensure 'invalid' is included
        if "invalid" not in all_cats:
            all_cats.append("invalid")
        
        # Create shared categorical dtype for this group
        cat_dtype = pd.CategoricalDtype(categories=all_cats, ordered=False)
        invalid_code = all_cats.index("invalid")
        
        # Encode with consistent mapping
        y_true = group["ground_truth_text"].astype(cat_dtype).cat.codes.astype(np.int16).to_numpy()
        y_pred = group["prediction_text"].astype(cat_dtype).cat.codes.astype(np.int16).to_numpy()
        
        # IMPORTANT: label set is per-group (dataset/benchmark/model)
        label_codes_group = np.unique(np.concatenate([y_true]))#, y_pred]))
        label_codes_group = label_codes_group[label_codes_group != invalid_code]
        
        # If after filtering there are no labels, skip safely
        if label_codes_group.size == 0:
            continue
        
        group_seed = int(main_rng.integers(0, 2**32))
        rng = np.random.default_rng(group_seed)
        indices = rng.integers(0, len(y_true), size=(n_boot, len(y_true)))
        
        y_true_b = y_true[indices]
        y_pred_b = y_pred[indices]
        
        group_info = {"id_type": g_id[0], "model": g_id[1]}
        
        for m_type in ["precision", "recall"]:
            all_tasks.append((group_info, y_true_b, y_pred_b, label_codes_group, m_type))
    
    print(f"Executing Bootstrap on {len(all_tasks)} tasks across {n_jobs} cores...")
    results = Parallel(n_jobs=n_jobs)(
        delayed(_single_bootstrap_task_macro)(*t) for t in all_tasks
    )
    
    return pd.DataFrame(results)

In [26]:

def _permutation_worker_macro(task, n_perms, seed):
    rng = np.random.default_rng(seed)
    yt, yp1, yp2 = task["yt"], task["yp1"], task["yp2"]
    label_codes, m_type = task["label_codes"], task["metric"]

    obs1 = _vectorized_macro_metric_calc(yt, yp1, label_codes, m_type)
    obs2 = _vectorized_macro_metric_calc(yt, yp2, label_codes, m_type)
    obs_diff = obs1 - obs2
    
    # print(task['dataset'], task['model1'], task['model2'], obs1, obs2, obs_diff, len(yt), len(yp1), len(yp2))

    # Vectorized swap across all permutations
    swap = rng.integers(0, 2, size=(n_perms, len(yt)), dtype=bool)
    p1 = np.where(swap, yp2, yp1)
    p2 = np.where(swap, yp1, yp2)

    null1 = _vectorized_macro_metric_calc(yt, p1, label_codes, m_type)
    null2 = _vectorized_macro_metric_calc(yt, p2, label_codes, m_type)

    p_val = float(np.mean(np.abs(null1 - null2) >= np.abs(obs_diff)))

    out = {k: v for k, v in task.items() if k not in ["yt", "yp1", "yp2", "label_codes"]}
    out.update(
        {
            "p_value": p_val,
            "observed_diff": float(obs_diff),
            "metric": f"macro_{m_type}",
            "obs1": obs1,
            "obs2": obs2
        }
    )
    return out



def compute_pairwise_comparisons_macro(df, n_permutations=1000, seed=42, n_jobs=-1):
    """
    Permutation tests on macro precision/recall between model pairs within each (dataset, benchmark).
    Macro labels are computed per (dataset, benchmark) using union(y_true, y_pred1, y_pred2), excluding 'invalid'.
    Applies BH correction.
    """

    # Keep original text columns for grouping
    df_grouped = df[["ID", "trial", "dataset", "benchmark", "model", "ground_truth_text", "prediction_text", "id_type"]].copy()

    tasks = []
    for id_type, group in df_grouped.groupby("id_type", observed=True):
        # Get all unique categories for THIS dataset/benchmark
        gt_cats = group["ground_truth_text"].astype("category").cat.categories.tolist()
        pred_cats = group["prediction_text"].astype("category").cat.categories.tolist()
        all_cats = sorted(set(gt_cats + pred_cats))
        
        # Ensure 'invalid' is included
        if "invalid" not in all_cats:
            all_cats.append("invalid")
        
        # Create shared categorical dtype for this dataset/benchmark
        cat_dtype = pd.CategoricalDtype(categories=all_cats, ordered=False)
        invalid_code = all_cats.index("invalid")
        
        # Encode with consistent mapping
        group_int = pd.DataFrame({
            "ID": group["ID"],
            "trial": group["trial"],
            "model": group["model"],
            "y_true": group["ground_truth_text"].astype(cat_dtype).cat.codes.astype(np.int16),
            "y_pred": group["prediction_text"].astype(cat_dtype).cat.codes.astype(np.int16),
        })
        
        models = sorted(group_int["model"].unique())

        for m1, m2 in combinations(models, 2):
            d1 = group_int[group_int["model"] == m1].sort_values(['ID', 'trial']).reset_index(drop=True)
            d2 = group_int[group_int["model"] == m2].sort_values(['ID', 'trial']).reset_index(drop=True)
            if len(d1) != len(d2):
                continue  # needs paired alignment

            yt = d1["y_true"].to_numpy()
            yp1 = d1["y_pred"].to_numpy()
            yp2 = d2["y_pred"].to_numpy()

            # IMPORTANT: label set per dataset/benchmark
            label_codes = np.unique(np.concatenate([yt]))#, yp1, yp2]))
            label_codes = label_codes[label_codes != invalid_code]
            if label_codes.size == 0:
                continue

            for m_type in ["precision", "recall"]:
                tasks.append(
                    {
                        "id_type": id_type,
                        "model1": m1,
                        "model2": m2,
                        "metric": m_type,
                        "label_codes": label_codes,
                        "yt": yt,
                        "yp1": yp1,
                        "yp2": yp2,
                    }
                )

    print(f"Executing Permutation Tests on {len(tasks)} macro tasks...")
    seeds = np.random.default_rng(seed).integers(0, 2**32, size=len(tasks))
    results = Parallel(n_jobs=n_jobs)(
        delayed(_permutation_worker_macro)(tasks[i], n_permutations, int(seeds[i]))
        for i in range(len(tasks))
    )

    res_df = pd.DataFrame(results)
    if len(res_df) > 0:
        _, res_df["p_value_bh"], _, _ = multipletests(res_df["p_value"], method="fdr_bh")
    else:
        res_df["p_value_bh"] = []

    return res_df

In [21]:
# Step 1: Compute bootstrap CIs for macro metrics
all_macro_metrics = optimized_bootstrap_parallel_macro(
    results_df, n_boot=1000, seed=42, n_jobs=20
).sort_values(["id_type", "model", "metric"])

Preparing Bootstrap data for 12 macro tasks...
Executing Bootstrap on 12 tasks across 20 cores...


In [22]:
all_macro_metrics

,id_type,model,metric,mean,median,low,high
0,MORE_THAN_ONE,NACC-3B-OS-SCE,macro_precision,0.242285,0.242296,0.236962,0.247762
1,MORE_THAN_ONE,NACC-3B-OS-SCE,macro_recall,0.396637,0.396637,0.383011,0.410318
2,MORE_THAN_ONE,Qwen2.5-3B-Instruct,macro_precision,0.203064,0.203165,0.197351,0.208231
3,MORE_THAN_ONE,Qwen2.5-3B-Instruct,macro_recall,0.249270,0.249394,0.239069,0.260063
4,MORE_THAN_ONE,Qwen2.5-7B-Instruct,macro_precision,0.252004,0.251924,0.243431,0.260881
5,MORE_THAN_ONE,Qwen2.5-7B-Instruct,macro_recall,0.294739,0.294712,0.283792,0.307071
6,ONE,NACC-3B-OS-SCE,macro_precision,0.272904,0.272883,0.269412,0.276631
7,ONE,NACC-3B-OS-SCE,macro_recall,0.467873,0.467792,0.459019,0.478109
8,ONE,Qwen2.5-3B-Instruct,macro_precision,0.219459,0.219469,0.216022,0.222944
9,ONE,Qwen2.5-3B-Instruct,macro_recall,0.274052,0.274156,0.266801,0.280751


In [27]:
# Step 2: Compute pairwise permutation comparisons on macro metrics
pairwise_macro = compute_pairwise_comparisons_macro(
    results_df, n_permutations=1000, seed=42, n_jobs=20
).sort_values(["id_type", "metric", "model1", "model2"])

Executing Permutation Tests on 12 macro tasks...


In [28]:
pairwise_macro

,id_type,model1,model2,metric,p_value,observed_diff,obs1,obs2,p_value_bh
0,MORE_THAN_ONE,NACC-3B-OS-SCE,Qwen2.5-3B-Instruct,macro_precision,0.000,0.039160,0.242292,0.203132,0.000
2,MORE_THAN_ONE,NACC-3B-OS-SCE,Qwen2.5-7B-Instruct,macro_precision,0.003,-0.009838,0.242292,0.252130,0.003
4,MORE_THAN_ONE,Qwen2.5-3B-Instruct,Qwen2.5-7B-Instruct,macro_precision,0.000,-0.048998,0.203132,0.252130,0.000
1,MORE_THAN_ONE,NACC-3B-OS-SCE,Qwen2.5-3B-Instruct,macro_recall,0.000,0.147571,0.397054,0.249484,0.000
3,MORE_THAN_ONE,NACC-3B-OS-SCE,Qwen2.5-7B-Instruct,macro_recall,0.000,0.102139,0.397054,0.294916,0.000
5,MORE_THAN_ONE,Qwen2.5-3B-Instruct,Qwen2.5-7B-Instruct,macro_recall,0.000,-0.045432,0.249484,0.294916,0.000
6,ONE,NACC-3B-OS-SCE,Qwen2.5-3B-Instruct,macro_precision,0.000,0.053467,0.272877,0.219410,0.000
8,ONE,NACC-3B-OS-SCE,Qwen2.5-7B-Instruct,macro_precision,0.000,-0.028836,0.272877,0.301713,0.000
10,ONE,Qwen2.5-3B-Instruct,Qwen2.5-7B-Instruct,macro_precision,0.000,-0.082303,0.219410,0.301713,0.000
7,ONE,NACC-3B-OS-SCE,Qwen2.5-3B-Instruct,macro_recall,0.000,0.193596,0.467666,0.274070,0.000


The following function both computes bootstrap stats on precision and recall (used in the figure later), and returns the raw bootstrap samples. 

The raw samples are used to compute the F1 score, and are available if we later want to compute something else. The F1 score is only presented in the table, not in the figure.

In [29]:
# Define mappings
model_map = {
    "Qwen2.5-3B-Instruct": "Q3B",
    "NACC-3B-OS-SCE": "LUNAR",
    "Qwen2.5-7B-Instruct": "Q7B",
}

## Latex

In [72]:
import numpy as np
import pandas as pd

def generate_latex_table_macro(all_metric, model_map, dataset_map=None):
    """
    Builds a LaTeX table for macro_precision / macro_recall from the macro bootstrap output,
    and computes macro_f1 from those.

    Expected columns in all_metric:
      ['dataset','benchmark','model','metric','median','low','high']  (benchmark optional)
    where metric in {'macro_precision','macro_recall'} (or 'macro_precision'/'macro_recall').

    - model_map: dict mapping raw model -> display name
    - dataset_map: optional dict mapping raw dataset -> display name
    """
    df = all_metric.copy()

    # 1) Apply mappings
    df["model"] = df["model"].map(model_map).fillna(df["model"])
    if dataset_map is not None and "dataset" in df.columns:
        df["dataset"] = df["dataset"].map(dataset_map).fillna(df["dataset"])

    # 2) Pivot to get macro metrics as columns for F1 calculation
    idx_cols = ["dataset", "model"]
    if "benchmark" in df.columns:
        idx_cols.insert(1, "benchmark")  # dataset, benchmark, model

    stats = df.pivot_table(
        index=idx_cols,
        columns="metric",
        values=["median", "low", "high"]
    ).reset_index()

    # Flatten multi-index columns: ('median','macro_precision') -> 'macro_precision_median'
    stats.columns = [
        f"{col[1]}_{col[0]}" if col[1] else col[0]
        for col in stats.columns
    ]

    # 3) Calculate macro F1 for median and CI bounds
    # Use the same "compute F1 from precision/recall at each bound" approach you had.
    for stat in ["median", "low", "high"]:
        p = stats[f"macro_precision_{stat}"]
        r = stats[f"macro_recall_{stat}"]
        stats[f"macro_f1_{stat}"] = np.where((p + r) > 0, 2 * (p * r) / (p + r), 0.0)

    # 4) Optional: set a model sort order (edit as needed)
    model_order = ["Q3B", "LUNAR", "Q7B"]
    stats["model"] = pd.Categorical(stats["model"], categories=model_order, ordered=True)

    sort_cols = ["dataset", "model"]
    if "benchmark" in stats.columns:
        sort_cols = ["dataset", "benchmark", "model"]
    stats = stats.sort_values(by=sort_cols).reset_index(drop=True)

    # 5) Identify best model per (dataset[, benchmark]) for each metric (highest median)
    metrics = ["macro_precision", "macro_recall", "macro_f1"]
    group_cols = ["dataset"]
    if "benchmark" in stats.columns:
        group_cols.append("benchmark")

    best_lookup = set()
    for m in metrics:
        idx = stats.groupby(group_cols, observed=False)[f"{m}_median"].idxmax()
        for i in idx:
            if pd.notna(i):
                best_lookup.add((int(i), m))

    # 6) Build LaTeX
    headers = ["Dataset"]
    if "benchmark" in stats.columns:
        headers.append("Benchmark")
    headers += ["Model", "Macro Precision", "Macro Recall", "Macro F1"]

    colspec = "ll" if "benchmark" in stats.columns else "l"
    colspec += "lccc"  # Model + 3 metrics

    latex_lines = [
        "\\begin{table}[ht]",
        "\\centering",
        "\\small",
        f"\\begin{{tabular}}{{{colspec}}}",
        "\\toprule",
        " & ".join(headers) + " \\\\",
        "\\midrule"
    ]

    prev_group = None
    for i, row in stats.iterrows():
        # Add midrule when dataset (or dataset+benchmark) changes
        this_group = tuple(row[c] for c in group_cols)
        if prev_group is not None and this_group != prev_group:
            latex_lines.append("\\midrule")

        # Display dataset/benchmark only on first row of each group
        ds_disp = row["dataset"] if prev_group is None or row["dataset"] != prev_group[0] else ""
        parts = [ds_disp]

        if "benchmark" in stats.columns:
            bench_disp = row["benchmark"] if prev_group is None or (len(prev_group) > 1 and row["benchmark"] != prev_group[1]) else ""
            parts.append(bench_disp)

        parts.append(str(row["model"]))

        formatted_metrics = []
        for m in metrics:
            val_str = f"{row[f'{m}_median']:.3f} [{row[f'{m}_low']:.3f}, {row[f'{m}_high']:.3f}]"
            if (i, m) in best_lookup:
                val_str = f"\\textbf{{{val_str}}}"
            formatted_metrics.append(val_str)

        row_str = " & ".join(parts) + " & " + " & ".join(formatted_metrics) + " \\\\"
        latex_lines.append(row_str)

        prev_group = this_group

    latex_lines.extend(["\\bottomrule", "\\end{tabular}", "\\end{table}"])
    return "\n".join(latex_lines)


In [ ]:
latex_output = generate_latex_table_macro(all_macro_metrics, model_map=model_map)#, class_map=class_map)
print(latex_output)

## Bar plot

In [30]:



def get_significance_marker(p_value):
    if isinstance(p_value, str):
        return p_value
    if pd.isna(p_value):
        return ""
    if p_value < 0.0001:
        return "****"
    elif p_value < 0.001:
        return "***"
    elif p_value < 0.01:
        return "**"
    elif p_value < 0.05:
        return "*"
    elif p_value <= 1.0:
        return "ns"
    else:
        return str(p_value)


def plot_macro_with_pvalues(
    all_metrics, pairwise_pvalues,
    model_map,
    output_dir=".",
    show_all_comparisons=False,
    p_threshold=0.05,
    model_order=("Q3B", "LUNAR", "Q7B"),
    order=("ONE", "MORE_THAN_ONE"),
):
    """
    Plot MACRO precision and recall with p-value annotations.

    all_metrics: output of optimized_bootstrap_parallel_macro(...)
      columns: dataset, benchmark(optional), model, metric, median, low, high
      metric: 'macro_precision', 'macro_recall'

    pairwise_pvalues: output of compute_pairwise_comparisons_macro(...)
      columns: dataset, benchmark(optional), model1, model2, metric, p_value, p_value_bh, observed_diff
      metric: 'macro_precision', 'macro_recall'   (as returned in the earlier macro code)
    """

    # -----------------------
    # Apply mappings
    # -----------------------
    df = all_metrics.copy()
    df["model_abbrev"] = df["model"].map(model_map).fillna(df["model"])

    pv = pairwise_pvalues.copy()
    pv["model1_abbrev"] = pv["model1"].map(model_map).fillna(pv["model1"])
    pv["model2_abbrev"] = pv["model2"].map(model_map).fillna(pv["model2"])

    # Keep only macro precision/recall
    df = df[df["metric"].isin(["macro_precision", "macro_recall"])].copy()
    pv = pv[pv["metric"].isin(["macro_precision", "macro_recall"])].copy()

    # Determine if benchmark exists
    has_benchmark = ("benchmark" in df.columns) and ("benchmark" in pv.columns)

    # Dataset list
    ids_present = sorted(df["id_type"].unique())
    id_types = [d for d in order if d in ids_present] or ids_present
    
    # datasets = sorted(df["dataset"].unique())

    # Models (fixed)
    models = list(model_order)
    palette = dict(zip(models, sns.color_palette("colorblind", n_colors=len(models))))

    # Two rows: macro_precision, macro_recall; columns: datasets
    metrics = ["macro_precision", "macro_recall"]
    metric_names = {"macro_precision": "Macro Precision", "macro_recall": "Macro Recall"}
    n_id_types = len(id_types)

    fig, axes = plt.subplots(2, n_id_types, figsize=(4 * n_id_types, 6))
    if n_id_types == 1:
        axes = axes.reshape(-1, 1)

    n_models = len(models)
    bar_width = 0.8 / n_models

    # -----------------------
    # Plot panels
    # -----------------------
    for col_idx, id_type in enumerate(id_types):
        for row_idx, metric in enumerate(metrics):
            ax = axes[row_idx, col_idx]

            # Filter to this panel
            panel = df[(df["id_type"] == id_type) & (df["metric"] == metric)].copy()

            # If benchmark exists, you probably want exactly one benchmark per dataset here.
            # We aggregate across benchmarks by taking the first if multiple exist.
            # If you want faceting by benchmark instead, say so and I’ll refactor.
            if has_benchmark and panel["benchmark"].nunique() > 1:
                first_bench = sorted(panel["benchmark"].unique())[0]
                panel = panel[panel["benchmark"] == first_bench]

            # Bar positions (single x category called "Macro")
            x_center = 0
            bar_info = {m: {"x": None, "height": 0.0} for m in models}

            # Plot each model bar
            for i, model in enumerate(models):
                mdata = panel[panel["model_abbrev"] == model]

                if len(mdata) == 0:
                    val, low, high = 0.0, 0.0, 0.0
                else:
                    val = float(mdata["median"].iloc[0])
                    low = float(mdata["low"].iloc[0])
                    high = float(mdata["high"].iloc[0])

                err_low = val - low
                err_high = high - val

                x_pos = x_center + i * bar_width - (n_models - 1) * bar_width / 2

                bar_info[model]["x"] = x_pos
                bar_info[model]["height"] = val + err_high

                bars = ax.bar(
                    [x_pos],
                    [val],
                    bar_width,
                    label=model if (row_idx == 0 and col_idx == n_id_types - 1) else "",
                    color=palette[model],
                    alpha=0.8,
                    yerr=[[err_low], [err_high]],
                    capsize=3,
                    error_kw={"linewidth": 1.5},
                )

                # Value annotation above error bar
                if val > 0:
                    ax.text(
                        x_pos,
                        val + err_high + 0.02,
                        f"{val:.2f}",
                        ha="center",
                        va="bottom",
                        fontsize=8,
                        color="black",
                    )

            # -----------------------
            # P-value annotations
            # -----------------------
            psubset = pv[
                (pv["id_type"] == id_type)
                & (pv["metric"] == metric)
                & (pv["p_value_bh"] < p_threshold)
            ].copy()

            if has_benchmark and "benchmark" in psubset.columns and panel["benchmark"].nunique() == 1:
                b = panel["benchmark"].iloc[0]
                psubset = psubset[psubset["benchmark"] == b]

            if not show_all_comparisons:
                baseline_model = models[0]
                psubset = psubset[
                    (psubset["model1_abbrev"] == baseline_model)
                    | (psubset["model2_abbrev"] == baseline_model)
                ]

            psubset = psubset.sort_values("p_value_bh")

            max_bar_height = max(bar_info[m]["height"] for m in models)

            y_offset = 0.15
            y_step = 0.12

            for comp_idx, (_, prow) in enumerate(psubset.iterrows()):
                m1 = prow["model1_abbrev"]
                m2 = prow["model2_abbrev"]
                pval = float(prow["p_value_bh"])

                if (m1 not in models) or (m2 not in models):
                    continue

                x1 = bar_info[m1]["x"]
                x2 = bar_info[m2]["x"]
                if x1 is None or x2 is None:
                    continue

                y_bracket = max_bar_height + y_offset + comp_idx * y_step
                p_text = get_significance_marker(pval)

                ax.plot(
                    [x1, x1, x2, x2],
                    [y_bracket - 0.01, y_bracket, y_bracket, y_bracket - 0.01],
                    "k-",
                    linewidth=1,
                )
                ax.text(
                    (x1 + x2) / 2,
                    y_bracket + 0.005,
                    p_text,
                    ha="center",
                    va="bottom",
                    fontsize=8,
                    fontweight="bold",
                )

            # -----------------------
            # Formatting
            # -----------------------
            ax.set_ylabel(metric_names[metric], fontsize=10)
            if row_idx == 0:
                ax.set_title(f"{id_type}", fontsize=11)

            ax.set_xticks([0])
            ax.set_xticklabels([""], rotation=0)

            # Adjust y-limit for brackets
            max_brackets = len(psubset)
            max_y = 1.2 + max_brackets * y_step if max_brackets > 0 else 1.2
            ax.set_ylim(0, max_y)

            yticks = ax.get_yticks()
            yticks_filtered = yticks[yticks <= 1.0]
            ax.set_yticks(yticks_filtered)
            ax.set_yticklabels([f"{y:.1f}" for y in yticks_filtered])

            ax.grid(True, alpha=0.3, axis="y")
            ax.grid(False, axis="x")

            # sns.despine(ax=ax, left=True, bottom=True, right=True, top=True)
            # if id_type == "NACC":
            #     for spine in ["left", "bottom", "right", "top"]:
            #         ax.spines[spine].set_visible(True)
            #         ax.spines[spine].set_linewidth(1.0)

    # Legend
    handles = [plt.Rectangle((0, 0), 1, 1, fc=palette[m], alpha=0.8, label=m) for m in models]
    fig.legend(
        handles=handles,
        title="Model",
        loc="lower center",
        ncol=len(models),
        bbox_to_anchor=(0.5, -0.053),
        frameon=True,
    )

    # Optional custom titles (only if exactly those panels exist)
    if n_id_types >= 2:
        axes[0, 0].set_title("One etiology")
        axes[0, 1].set_title("More than one etiologies")

    plt.tight_layout(rect=[0, 0.02, 1, 1])

    # Save
    os.makedirs(output_dir, exist_ok=True)
    filename = os.path.join(output_dir, "../figures/fig3_nacc.pdf")
    plt.savefig(filename, bbox_inches="tight", dpi=300)
    print(f"Saved {filename}")
    plt.close()




In [31]:
# all_macro_metrics['dataset'].unique()

In [32]:
# dataset_order = ['NACC', 'NIFD', 'PPMI', 'ADNI', 'BrainLat']

In [33]:
# # Example call (macro)
plot_macro_with_pvalues(
    all_metrics=all_macro_metrics,          # from optimized_bootstrap_parallel_macro(...)
    pairwise_pvalues=pairwise_macro,        # from compute_pairwise_comparisons_macro(...)
    model_map=model_map,
    output_dir=".",
    show_all_comparisons=True,
    p_threshold=1,  # set <0.05 to filter
)


Saved ./../figures/fig3_nacc.pdf
